# 16 — Sea-level-rise risk: SaTScan clusters → GeoAI story map

![PBIX-derived six perspectives](assets/six-thinking-hats-global-health.png)

This lab turns the SaTScan cluster summary embedded in `Prototype-July-8-2024.pbix` into a browser-first GeoAI investigation. The report’s `ALLMODELS` table supplies cluster rank, flood facilities, observed/expected ratio, relative risk, radius, and significance. Its `Climate-Sites-w-coords` table supplies city coordinates, elevation context, and scenario flood indicators.

This is a decision-support teaching example—not a new SaTScan run, a forecast, or a validated flood-hazard map.

In [ ]:
import sys
if sys.platform == 'emscripten':
    import micropip
    await micropip.install(['geolibre==3.0.0', 'pyodide-http', 'plotly==6.5.0'])
    import pyodide_http
    pyodide_http.patch_all()
from geolibre_lite import LiteMap as Map
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import plotly.express as px
import plotly.graph_objects as go
risk = pd.read_csv('data/pbix_top10_sat_scan.csv')
risk['city_label'] = risk['city_name'] + ', ' + risk['country']
risk


## Story map 1 — What does the PBIX evidence say?

SaTScan identifies unusually concentrated spatial risk. Rank and relative risk are evidence about reported clusters; they are not interchangeable with meters of water, population exposed, or a probability that a particular building floods.

In [ ]:
fig = px.bar(risk.sort_values('relative_risk'), x='relative_risk', y='city_label', orientation='h', color='flood_facilities', color_continuous_scale='Turbo', labels={'relative_risk':'SaTScan relative risk (report units)','city_label':'City','flood_facilities':'Flood facilities'})
fig.update_layout(height=560, title='Reported SaTScan clusters: relative risk and flood-facility count')
fig.show()
m = Map(center=(35, 35), zoom=1.3, height='650px')
features = [{'type':'Feature','geometry':{'type':'Point','coordinates':[r.longitude,r.latitude]},'properties':{'city':r.city_label,'cluster_rank':int(r.cluster_rank),'relative_risk':float(r.relative_risk),'flood_facilities':int(r.flood_facilities)}} for r in risk.itertuples()]
m.add_geojson({'type':'FeatureCollection','features':features}, name='PBIX SaTScan top 10', radius=8, fillColor='#e45756', fillOpacity=0.8, strokeColor='#351c75', strokeWidth=1.5)
m


## Story map 2 — Scenario animation

The PBIX includes three scenario columns summarized here as A, B, and C. The animation is a sense-making device: it asks where the pattern changes most without pretending that a scenario value is direct water depth.

In [ ]:
scenario_cols = {'A':'flood_a_mean','B':'flood_b_mean','C':'flood_c_mean'}
long = risk.melt(id_vars=['city_label','relative_risk'], value_vars=list(scenario_cols.values()), var_name='scenario', value_name='scenario_indicator')
long['scenario'] = long['scenario'].map({v:k for k,v in scenario_cols.items()})
fig = px.scatter(long, x='relative_risk', y='city_label', animation_frame='scenario', size='scenario_indicator', color='scenario_indicator', color_continuous_scale='Viridis', hover_name='city_label', labels={'relative_risk':'SaTScan relative risk','city_label':'City','scenario_indicator':'PBIX scenario indicator'})
fig.update_layout(height=620, title='Animated scenario comparison — PBIX flood indicator, not water depth')
fig.show()


## Story map 3 — 3D risk and a responsible GeoAI deep dive

A 3D view can show several dimensions at once, but it can also create false confidence. Height encodes relative risk, point size encodes facilities, and color encodes the all-scenario indicator. Always compare it with the table and 2D map.

In [ ]:
fig3d = go.Figure(go.Scatter3d(x=risk.longitude, y=risk.latitude, z=risk.relative_risk, mode='markers+text', text=risk.city_name, textposition='top center', marker={'size':np.clip(risk.flood_facilities/18,6,18),'color':risk.flood_all_mean,'colorscale':'Viridis','showscale':True,'colorbar':{'title':'All-scenario indicator'}}, hovertemplate='%{text}<br>relative risk=%{z:.1f}<extra></extra>'))
fig3d.update_layout(height=680, title='3D sense-making view: location × relative risk × reported facilities', scene={'xaxis_title':'Longitude','yaxis_title':'Latitude','zaxis_title':'Relative risk'})
fig3d.show()


## Six perspectives for the next decision

- **White / evidence:** Which source field, date, denominator, and spatial unit support each claim?
- **Red / experience:** What would residents, responders, and health workers notice that the table cannot show?
- **Black / caution:** Where could missing data, geocoding, model assumptions, or map design harm a community?
- **Yellow / opportunity:** Which early-warning, planning, or service-access decision could this evidence improve?
- **Green / alternatives:** What other data, methods, or community-led interpretations should challenge the first model?
- **Blue / process:** Who owns the next action, what gets validated, and when do we stop using the model?
- **Purple / synthesis:** What is the smallest accountable next step that preserves disagreement and makes uncertainty visible?

A useful GeoAI workflow is **discover → join → engineer → compare → explain → challenge**. The PBIX SaTScan result identifies reported spatial clusters; the scenario fields provide a comparative teaching layer. The next responsible step is validation with local data and accountable partners—not a stronger claim.

## Data & software citations

- User-supplied Power BI report: `Prototype-July-8-2024.pbix`. The committed CSV is a derived top-10 summary, not the complete report model.
- SaTScan-derived report table: `ALLMODELS`; supporting table: `Climate-Sites-w-coords`.
- [GeoLibre](https://geolibre.app/), [GeoAI](https://opengeoai.org/), and [Plotly Python](https://plotly.com/python/).
- Six Thinking Hats concepts are associated with Edward de Bono; the purple synthesis framing and supplied graphic are retained as teaching context.
- Preserve the report’s original source citations and licensing before publishing any derivative result.